EDA
 ↓
Seasonality adjustment / feature engineering
 ↓
Dense AE
 ↓
LSTM AE
 ↓
LSTM/GRU forecasting anomaly detector
 ↓
Compare anomalies
 ↓
Examine specific dates flagged by each model

#Load historical Dataset

In [2]:
import pandas as pd
import numpy as np

triangle_daily = pd.read_parquet('data/triangle_pm25_daily_2021_2025.parquet')


| Model                             | What it learns                                 | How anomaly score works             | Worth trying?                  |
| --------------------------------- | ---------------------------------------------- | ----------------------------------- | ------------------------------ |
| **Dense Autoencoder**             | Normal combinations of weather variables       | Reconstruction error                | ✅ Definitely                   |
| **LSTM Autoencoder**              | Normal multi-day weather sequences             | Sequence reconstruction error       | ✅ Definitely                   |
| **Forecasting LSTM/GRU**          | Predicts the next day from prior days          | Prediction residual                 | ✅ Probably my favorite         |
| **Variational Autoencoder (VAE)** | Probabilistic representation of normal weather | Reconstruction + latent probability | ✅ Good advanced model          |
| **1D CNN Autoencoder**            | Short-term local temporal patterns             | Reconstruction error                | 🟡 Interesting comparison      |
| **Transformer Autoencoder**       | Longer-range sequence relationships            | Reconstruction error                | 🟡 Cool, but probably overkill |


Encode sinusoidal seasonality function, PM 2.5 and ozone both have seasonality factors

In [3]:
air = triangle_daily.copy()

air["date"] = pd.to_datetime(air["date"])
air = air.sort_values("date").reset_index(drop=True)

# Basic calendar variables
air["year"] = air["date"].dt.year
air["month"] = air["date"].dt.month
air["day_of_year"] = air["date"].dt.dayofyear
air["day_of_week"] = air["date"].dt.dayofweek
air["is_weekend"] = (air["day_of_week"] >= 5).astype(int)

# Meteorological season - useful mainly for plots / interpretation
air["season"] = air["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
})

# Cyclical annual seasonality
air["doy_sin"] = np.sin(
    2 * np.pi * air["day_of_year"] / 365.25
)

air["doy_cos"] = np.cos(
    2 * np.pi * air["day_of_year"] / 365.25
)

# Cyclical weekly pattern
air["dow_sin"] = np.sin(
    2 * np.pi * air["day_of_week"] / 7
)

air["dow_cos"] = np.cos(
    2 * np.pi * air["day_of_week"] / 7
)

The data is sequential, not independent or identically distributed. So adding lag to account for previous day's PM 2.5 values

In [4]:
pm25_lags = [1, 2, 3, 7, 14]

for lag in pm25_lags:
    air[f"pm25_lag_{lag}"] = air["pm25"].shift(lag)

In [5]:
for window in [3, 7, 14, 30]:

    # shift first so today's PM2.5 is NOT included
    historical_pm25 = air["pm25"].shift(1)

    air[f"pm25_mean_{window}d"] = (
        historical_pm25
        .rolling(window)
        .mean()
    )

    air[f"pm25_std_{window}d"] = (
        historical_pm25
        .rolling(window)
        .std()
    )

Short-term change

In [6]:
air["pm25_change_1d"] = air["pm25"] - air["pm25_lag_1"]
air["pm25_change_7d"] = air["pm25"] - air["pm25_lag_7"]

# Import ozone data and train dense auto encoder

In [16]:
import pandas as pd
import numpy as np

air = pd.read_parquet(
    "data/triangle_pm25_ozone_daily_2021_2025.parquet"
)

air["date"] = pd.to_datetime(air["date"])
air = air.sort_values("date").reset_index(drop=True)

# Annual seasonality
air["day_of_year"] = air["date"].dt.dayofyear

air["doy_sin"] = np.sin(
    2 * np.pi * air["day_of_year"] / 365.25
)

air["doy_cos"] = np.cos(
    2 * np.pi * air["day_of_year"] / 365.25
)

Define feature columns

In [17]:
feature_cols = [
    "pm25",
    "ozone_8hr_max",
    "doy_sin",
    "doy_cos"
]

Split into training and test by date

In [18]:
train = air[
    air["date"] < "2025-01-01"
].copy()

val = air[
    air["date"] >= "2025-01-01"
].copy()

Check for missing values

In [19]:
print(
    air[feature_cols].isna().sum()
)

pm25              0
ozone_8hr_max    15
doy_sin           0
doy_cos           0
dtype: int64


Drop missing values from dataset

In [20]:
train_model = train.dropna(
    subset=feature_cols
).copy()

val_model = val.dropna(
    subset=feature_cols
).copy()

Fit the training and validation sets by shape w scaler

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train_model[feature_cols]
)

X_val = scaler.transform(
    val_model[feature_cols]
)

print(X_train.shape)
print(X_val.shape)

(1447, 4)
(364, 4)


Define shape and layers

In [22]:
import tensorflow as tf
from tensorflow.keras import layers, Model

n_features = X_train.shape[1]

inputs = layers.Input(shape=(n_features,))

encoded = layers.Dense(
    3,
    activation="relu"
)(inputs)

latent = layers.Dense(
    2,
    activation="relu",
    name="latent_space"
)(encoded)

decoded = layers.Dense(
    3,
    activation="relu"
)(latent)

outputs = layers.Dense(
    n_features,
    activation="linear"
)(decoded)

autoencoder = Model(inputs, outputs)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            15 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_space (Dense)            │ (None, 2)              │             8 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            16 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48 (192.00 B)

 Trainable params: 48 (192.00 B)

 Non-trainable params: 0 (0.00 B)

Train the autoencoder

In [23]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train,
    X_train,
    validation_data=(X_val, X_val),
    epochs=200,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 1.3371 - val_loss: 1.0240
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.1565 - val_loss: 0.9257
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.0594 - val_loss: 0.8705
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.0033 - val_loss: 0.8351
Epoch 5/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.9663 - val_loss: 0.8093
Epoch 6/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.9382 - val_loss: 0.7883
Epoch 7/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.9148 - val_loss: 0.7695
Epoch 8/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8944 - val_loss: 0.7529
Epoch 9/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8765 - val_loss: 0.7380
Epoch 10/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.8607 - val_loss: 0.7250
Epoch 11/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.8472 - val_loss: 0.7138
Epoch 12/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.

Create threshold for anomaly detection

In [24]:
train_reconstructed = autoencoder.predict(X_train)

train_error = np.mean(
    np.square(X_train - train_reconstructed),
    axis=1
)

threshold = np.percentile(
    train_error,
    95
)

print("Threshold:", threshold)

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Threshold: 1.122694049908566


Predict values on reconstructed validation data

In [25]:
val_reconstructed = autoencoder.predict(X_val)

val_error = np.mean(
    np.square(X_val - val_reconstructed),
    axis=1
)

val_model["reconstruction_error"] = val_error

val_model["dense_ae_anomaly"] = (
    val_error > threshold
).astype(int)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [26]:
feature_errors = np.square(
    X_val - val_reconstructed
)

pm25_idx = feature_cols.index("pm25")
ozone_idx = feature_cols.index("ozone_8hr_max")

val_model["pm25_reconstruction_error"] = (
    feature_errors[:, pm25_idx]
)

val_model["ozone_reconstruction_error"] = (
    feature_errors[:, ozone_idx]
)

Sort by error on features

In [27]:
val_model[
    [
        "date",
        "pm25",
        "ozone_8hr_max",
        "reconstruction_error",
        "pm25_reconstruction_error",
        "ozone_reconstruction_error",
        "dense_ae_anomaly"
    ]
].sort_values(
    "reconstruction_error",
    ascending=False
).head(20)

,date,pm25,ozone_8hr_max,reconstruction_error,pm25_reconstruction_error,ozone_reconstruction_error,dense_ae_anomaly
1785,2025-11-21,20.309722,13.0,3.653319,12.009882,2.191031,1
1680,2025-08-08,19.564583,36.0,2.022538,5.256301,1.639198,1
1518,2025-02-27,17.626389,51.0,1.727394,3.473218,0.063192,1
1532,2025-03-13,17.500397,51.0,1.554450,3.126003,0.013733,1
1684,2025-08-12,2.954861,18.5,1.496869,0.599487,1.257267,1
1801,2025-12-07,13.181945,11.0,1.361223,4.126977,1.248503,1
1647,2025-07-06,3.881944,23.5,1.290369,0.652147,1.340360,1
1533,2025-03-14,14.962500,33.0,1.281844,2.649978,0.458386,1
1608,2025-05-28,2.815972,26.5,1.256930,1.207144,0.882742,1
1607,2025-05-27,3.114070,26.0,1.241322,1.045488,0.956963,1
